In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Quantum Portfolio Optimizer: O Funcție Qiskit de la Global Data Quantum


*Vezi [referința API](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer)*

> **Note:** Funcțiile Qiskit sunt o funcționalitate experimentală disponibilă doar utilizatorilor planurilor IBM Quantum&reg; Premium, Flex și On-Prem (prin API-ul IBM Quantum Platform). Acestea se află în stadiu de previzualizare și pot suferi modificări.
## Prezentare generală
Quantum Portfolio Optimizer este o Funcție Qiskit care abordează problema optimizării dinamice a portofoliului, o problemă standard în finanțe ce urmărește reechilibrarea periodică a investițiilor într-un set de active, pentru a maximiza randamentele și a minimiza riscurile. Prin aplicarea tehnicilor de vârf din optimizarea cuantică, această funcție simplifică procesul astfel încât utilizatorii, fără expertiză în calculul cuantic, să poată beneficia de avantajele sale în identificarea traiectoriilor optime de investiție. Ideală pentru managerii de portofoliu, cercetătorii în finanțe cantitative și investitorii individuali, acest instrument permite back-testarea strategiilor de tranzacționare în optimizarea portofoliului.
## Descrierea funcției
Funcția Quantum Portfolio Optimizer folosește algoritmul Variational Quantum Eigensolver (VQE) pentru a rezolva o problemă de Optimizare Binară Neconstrasă Pătratică (QUBO), abordând probleme de optimizare dinamică a portofoliului. Utilizatorii trebuie doar să furnizeze datele privind prețurile activelor și să definească constrângerea de investiție, după care funcția rulează procesul de optimizare cuantică ce returnează un set de traiectorii de investiție optimizate.

Procesul constă în patru etape principale. Mai întâi, datele de intrare sunt mapate într-o problemă compatibilă cu calculul cuantic, construindu-se qubo-ul problemei de optimizare dinamică a portofoliului și transformându-l într-un operator cuantic (Hamiltonianul Ising). Apoi, problema de intrare și algoritmul VQE sunt adaptate pentru a fi rulate pe hardware cuantic. Algoritmul VQE este apoi rulat pe hardware cuantic, iar în final, rezultatele sunt post-procesate pentru a furniza traiectoriile optime de investiție. Sistemul include, de asemenea, o post-procesare conștientă de zgomot (bazată pe [SQD](/guides/qiskit-addons-sqd)) pentru a maximiza calitatea rezultatelor.

Această Funcție Qiskit se bazează pe [manuscrisul publicat](https://arxiv.org/abs/2412.19150) de Global Data Quantum.
![Vizualizarea fluxului de lucru al funcției](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)
# Începe
Autentifică-te folosind [cheia ta API](https://quantum.cloud.ibm.com/) și selectează Qiskit Function-ul după cum urmează. (Acest fragment presupune că ți-ai [salvat deja contul](/guides/functions#install-qiskit-functions-catalog-client) în mediul tău local.)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

## Exemplu: Optimizarea dinamică a portofoliului cu șapte active
Acest exemplu demonstrează cum să execuți funcția de optimizare dinamică a portofoliului (DPO) și cum să îi ajustezi setările pentru performanță optimă. Include pași detaliați pentru reglarea fină a parametrilor în vederea obținerii rezultatelor dorite.

Acest caz implică șapte active, patru pași de timp și patru qubit de rezoluție, rezultând un total de 112 qubiți necesari.
### 1. Citește activele incluse în portofoliu.
Dacă toate activele din portofoliu sunt stocate într-un folder la o cale specifică, le poți încărca într-un `pandas.DataFrame` și le poți converti în obiect de tip `dict` folosind funcția de mai jos.

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them
    into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

Pentru acest exemplu, am utilizat activele [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y) și [XS2239553048](https://finance.yahoo.com/quote/XS2239553048). Figura de mai jos ilustrează datele utilizate în acest exemplu, prezentând evoluția zilnică a prețului de închidere al activelor în perioada 1 ianuarie – 1 septembrie 2023.

În acest exemplu, pentru a asigura uniformitatea datelor, am completat zilele fără tranzacționare cu prețul de închidere din ziua disponibilă anterioară. Aplicăm acest pas deoarece activele selectate provin din piețe diferite cu zile de tranzacționare variate, fiind esențial să standardizăm setul de date pentru consistență.
![Visualization of the historical data of the assets](../docs/images/guides/global-data-quantum-optimizer/assets.avif)
### 2. Definește problema.
Definește specificațiile problemei configurând parametrii din dicționarul `qubo_settings`.

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

### 3. Definește setările Optimizer-ului și ale ansatz-ului (Opțional)
Opțional, poți defini cerințe specifice pentru procesul de optimizare, inclusiv alegerea Optimizer-ului și a parametrilor acestuia, precum și specificarea primitivei și a configurațiilor sale.

Pentru Tailored Ansatz, dimensiunea populației aleasă s-a bazat pe experimente anterioare care au arătat că această valoare produce o optimizare stabilă și eficientă.

În cazul Real Amplitudes Ansatz, poți urma o relație liniară între ``population_size`` și numărul de qubiți din Circuit. Ca regulă aproximativă orientativă, se recomandă utilizarea unui ``population_size`` **minim** ``~ 0.8 * n_qubits`` pentru ansatz-ul `real_amplitudes`.

Se preconizează că Optimized Real Amplitudes va avea o performanță de optimizare mai bună decât ansatz-ul Real Amplitudes. Totuși, numărul de variabile de optimizat în acest ansatz crește mult mai rapid decât în cazul Real Amplitudes (a se vedea [manuscrisul](https://arxiv.org/pdf/2412.19150)). Prin urmare, pentru probleme de mari dimensiuni, Optimized Real Amplitudes necesită mai multe execuții de circuit. Optimized Real Amplitudes este probabil util pentru probleme care necesită până la 100 de qubiți, însă se recomandă precauție la setarea parametrilor ``population_size``. Ca exemplu al acestei scalări a ``population_size``, tabelul anterior arată că pentru o problemă cu 84 de qubiți, Optimized Real Amplitudes necesită un ``population_size`` de 120, în timp ce pentru o problemă cu 56 de qubiți, un ``population_size`` de 40 este suficient.

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

De asemenea, este posibil să alegi un ansatz specific. Exemplul de mai jos folosește ansatz-ul ``'Tailored'``.

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

### 4. Run the problem.

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 5. Retrieve results
Funcția returnează un dicționar cu traiectoriile de investiții ordonate de la cea mai mică la cea mai mare valoare a funcției obiectiv (vezi secțiunea [Output](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer#output) din referința API). Acest set de rezultate permite identificarea traiectoriei cu cel mai mic cost și a evaluărilor corespunzătoare ale investițiilor. În plus, oferă posibilitatea de a analiza diferite traiectorii, facilitând selectarea celor care se aliniază cel mai bine cu nevoile sau obiectivele specifice. Această flexibilitate asigură că alegerile pot fi adaptate pentru a corespunde unei varietăți de preferințe sau scenarii.
Începe prin a prezenta strategia rezultată care a obținut cel mai mic cost obiectiv găsit în timpul procesului.

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

Afterwards, using the metadata, you can access the results of all the sampled strategies. You can thereby further analyze the alternative trajectories returned by the optimizer. To do this, read the dictionary stored in `dpo_result['metadata']['all_samples_metrics']`, which contains not only additional information about the optimal strategy, but also details of the other candidate strategies evaluated during the optimization.

The following example shows how to read this information using `pandas` to extract key metrics associated with the optimal strategy. These include Restriction Deviation, Sharpe Ratio, and the corresponding investment return.

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


Ulterior, folosind metadatele, poți accesa rezultatele tuturor strategiilor eșantionate. Astfel, poți analiza în continuare traiectoriile alternative returnate de Optimizer. Pentru a face acest lucru, citește dicționarul stocat în `dpo_result['metadata']['all_samples_metrics']`, care conține nu doar informații suplimentare despre strategia optimă, ci și detalii despre celelalte strategii candidate evaluate în timpul optimizării.

Următorul exemplu arată cum să citești aceste informații folosind `pandas` pentru a extrage valorile cheie asociate strategiei optime. Acestea includ Deviația de Restricție, Raportul Sharpe și rentabilitatea corespunzătoare a investiției.